# 1.2.3 Massive Benchmark

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.generators import Math2DGenerator
from src.noise import AdaptiveGaussianNoise
from src.shadows import BandShadow
from src.splitter import GapSplitter
from src.orchestrator import RegressionOrchestator
from src.models import (
    LinearRegressionModel, 
    PolynomialRegressionModel, 
    DecisionTreeModel, 
    RandomForestModel,
    SVRModel,
    NeuralNetworkModel
)

### 1. 50 Funtzioen Sorkuntza (The Generator Zoo)
Funtzio laguntzaile bat sortuko dugu 50 funtzio ezberdin parametrizatzeko.

In [ ]:
def generate_curve_zoo(n_funcs=50):
    curves = {}

    curves["Linear_Basic"] = lambda x: 0.5*x + 5
    curves["Quadratic_U"] = lambda x: 0.2*(x-10)**2 + 2
    curves["Quadratic_Down"] = lambda x: -0.1*(x-10)**2 + 20
    curves["Cubic"] = lambda x: 0.01*(x-10)**3 + 10
    for i in range(6):

        a, b, c = np.random.uniform(-0.5, 0.5, 3)
        curves[f"Poly_Mix_{i}"] = lambda x, a=a, b=b, c=c: a*(x-10)**2 + b*x + 10

    curves["Sine_Simple"] = lambda x: 5 * np.sin(x) + 10
    curves["Cosine_Fast"] = lambda x: 3 * np.cos(2*x) + 10
    curves["Dampened_Sine"] = lambda x: 10 * np.exp(-0.1*x) * np.sin(2*x) + 10
    curves["Chirp"] = lambda x: 3 * np.sin(0.1*x**2) + 10
    for i in range(6):
        freq = np.random.uniform(0.5, 3.0)
        phase = np.random.uniform(0, 3.14)
        curves[f"Sine_Rand_{i}"] = lambda x, f=freq, p=phase: 4 * np.sin(f*x + p) + 10

    curves["Exp_Growth"] = lambda x: 0.5 * np.exp(0.2*x)
    curves["Exp_Decay"] = lambda x: 20 * np.exp(-0.2*x)
    curves["Log_Growth"] = lambda x: 5 * np.log(x + 1) + 2
    curves["Sigmoid"] = lambda x: 10 / (1 + np.exp(-(x-10))) + 5
    curves["Gaussian_Bell"] = lambda x: 15 * np.exp(-0.1*(x-10)**2) + 2
    for i in range(5):
        rate = np.random.uniform(0.1, 0.4)
        curves[f"Exp_Var_{i}"] = lambda x, r=rate: 2 * np.exp(r * (x-5))

    curves["Step_Simple"] = lambda x: np.where(x > 10, 15, 5)
    curves["Abs_V"] = lambda x: 2 * np.abs(x - 10) + 2
    curves["Sawtooth"] = lambda x: 5 * (x % 3) + 2
    curves["Square_Wave"] = lambda x: 5 * np.sign(np.sin(x)) + 10
    for i in range(6):
        thresh = np.random.uniform(5, 15)
        curves[f"Step_Rand_{i}"] = lambda x, t=thresh: np.where(x > t, 18, 4)

    curves["Linear_plus_Sine"] = lambda x: 0.5*x + 3*np.sin(x) + 5
    curves["Parabola_plus_Sine"] = lambda x: 0.1*(x-10)**2 + 2*np.sin(2*x) + 5
    curves["Sinc"] = lambda x: 10 * np.sinc(x-10) + 5
    curves["Double_Gaussian"] = lambda x: 10*np.exp(-0.5*(x-5)**2) + 10*np.exp(-0.5*(x-15)**2)
    for i in range(6):

        curves[f"Complex_Rand_{i}"] = lambda x: 0.2*x + 2*np.sin(np.random.uniform(0.5, 2)*x)

    return curves

curve_zoo = generate_curve_zoo(50)
print(f"Generated {len(curve_zoo)} unique functions.")

### 2. Konfigurazioa

In [ ]:
N_SAMPLES = 400
X_RANGE = (0, 20)
NOISE = AdaptiveGaussianNoise(noise_ratio=0.1, seed=42)
SPLITTER = GapSplitter(test_size=0.2, random_state=42)

SHADOWS = {
    "Small (2u)": BandShadow('x', 9, 11),
    "Medium (6u)": BandShadow('x', 7, 13),
    "Large (10u)": BandShadow('x', 5, 15)
}

MODEL_FACTORIES = [
    (lambda: LinearRegressionModel(), "Linear"),
    (lambda: PolynomialRegressionModel(degree=2), "Poly-2"),
    (lambda: PolynomialRegressionModel(degree=5), "Poly-5"),
    (lambda: DecisionTreeModel(), "Tree"),
    (lambda: RandomForestModel(n_estimators=50), "RandomForest"),
    (lambda: SVRModel(kernel='rbf', C=50), "SVR-RBF"),
    
    (lambda: NeuralNetworkModel(hidden_layer_sizes=(10,), max_iter=1000), "NN-Small (10)"),
    
    (lambda: NeuralNetworkModel(hidden_layer_sizes=(64, 32), max_iter=2000), "NN-Medium (64,32)"),
    
    (lambda: NeuralNetworkModel(hidden_layer_sizes=(100, 100, 50), max_iter=2000), "NN-Deep (100,100,50)")
]

### 3. Exekuzioa

In [ ]:
results = []

print(f"Starting massive benchmark on {len(curve_zoo)} curves x {len(SHADOWS)} shadows x {len(MODEL_FACTORIES)} models...")
count = 0
total_iters = len(curve_zoo) * len(SHADOWS) * len(MODEL_FACTORIES)

for curve_name, func in curve_zoo.items():
    gen = Math2DGenerator(func, seed=42)
    
    for shadow_name, shadow in SHADOWS.items():
        
        for model_factory, model_name in MODEL_FACTORIES:
            model = model_factory()
            orchestrator = RegressionOrchestator(gen, SPLITTER, model)
            
            try:
                res = orchestrator.run(N_SAMPLES, X_RANGE, NOISE, shadow)
                mse_shadow = res.metrics.mse_test_shadow
            except Exception as e:
                print(f"Error in {curve_name} - {model_name}: {e}")
                mse_shadow = np.nan

            results.append({
                "Curve": curve_name,
                "Shadow": shadow_name,
                "Model": model_name,
                "MSE": mse_shadow
            })
            
            count += 1
            if count % 100 == 0:
                print(f"Progress: {count}/{total_iters} experiments done.")

df_results = pd.DataFrame(results)
print("Benchmark finished!")

### 4. Analisia

In [ ]:
df_results['Rank'] = df_results.groupby(['Curve', 'Shadow'])['MSE'].rank(ascending=True)

def show_leaderboard(shadow_size):
    print(f"\n=== LEADERBOARD: {shadow_size} ===")
    subset = df_results[df_results['Shadow'] == shadow_size]
    
    leaderboard = subset.groupby('Model').agg(
        Mean_Rank=('Rank', 'mean'),
        Top_1_Count=('Rank', lambda x: (x==1).sum()),
        Worst_Count=('Rank', lambda x: (x==9).sum())
    ).sort_values(by='Mean_Rank')
    
    display(leaderboard)
    
    plt.figure(figsize=(10, 4))
    sns.barplot(x=leaderboard.index, y=leaderboard['Mean_Rank'], palette="viridis")
    plt.title(f"Average Rank (Lower is Better) - {shadow_size}")
    plt.xticks(rotation=45)
    plt.ylabel("Mean Rank (1=Best)")
    plt.show()

show_leaderboard("Small (2u)")
show_leaderboard("Medium (6u)")
show_leaderboard("Large (10u)")

### 5. Ondorio Orokorrak
Zein da eredurik sendoena guztira?

In [ ]:
print("\n=== GLOBAL CHAMPION (All Shadows) ===")
global_board = df_results.groupby('Model')['Rank'].mean().sort_values()
display(global_board)

In [ ]:
from src.visualization import plot_experiment_results

def visualize_specific_experiment(curve_key, shadow_key, model_keys):
    print(f"\n--- VISUALIZING: {curve_key} @ {shadow_key} ---")

    func = curve_zoo[curve_key]
    shadow = SHADOWS[shadow_key]
    gen = Math2DGenerator(func, seed=42)

    for m_key in model_keys:

        factory = next(f for f, name in MODEL_FACTORIES if name == m_key)
        model = factory()

        orch = RegressionOrchestator(gen, SPLITTER, model)
        res = orch.run(N_SAMPLES, X_RANGE, NOISE, shadow)

        mse = res.metrics.mse_test_shadow
        plot_experiment_results(
            res.split_data, 
            res.predictions, 
            title=f"{curve_key} | {shadow_key}\nModel: {m_key} | MSE: {mse:.4f}"
        )

visualize_specific_experiment(
    "Double_Gaussian", 
    "Small (2u)", 
    ["Linear", "Poly-5", "NN-Deep (100,100,50)", "RandomForest"]
)

visualize_specific_experiment(
    "Sawtooth", 
    "Small (2u)", 
    ["Linear", "Poly-5", "NN-Deep (100,100,50)", "RandomForest"]
)

visualize_specific_experiment(
    "Quadratic_U", 
    "Large (10u)", 
    ["Poly-2", "NN-Deep (100,100,50)"]
)

visualize_specific_experiment(
    "Linear_Basic", 
     "Large (10u)", 
     ["NN-Deep (100,100,50)", "Poly-5"]
)